# 16 — Advanced Dictionary Data Structures: From Basics to Interview-Ready

## What This Notebook Covers

This is a ground-up tutorial on the data structures most commonly tested in ServiceTitan-style
HackerRank interviews. No prior knowledge of any of these assumed — we build every concept
from scratch before layering on complexity.

**Topics (in order):**
1. Python `dict` internals — what you actually need to know
2. `defaultdict`, `Counter`, `OrderedDict` from `collections`
3. **MultiMap** — multiple values per key (confirmed ServiceTitan pattern)
4. **Inverted Index** — flip a MultiMap inside out (Part 2 variant)
5. **LRU Cache** — Least Recently Used eviction (extremely common)
6. **LFU Cache** — Least Frequently Used (harder variant)
7. **Priority-Queue-backed dict** — for dispatch / scheduling problems
8. **Thread-safe wrappers** — what Part 2 of HackerRank often adds

**How to use this:** Read the markdown cells first, then run the code, then try the
exercises at the end of each section. Everything is commented to explain *why*, not just *what*.


---
## Part 1: Python `dict` — What You Actually Need to Know

Before building custom structures, you need to understand what a plain `dict` gives you
(and what it *doesn't*).

### The key facts:
- `dict` maps **one key → one value**. If you write `d["a"] = 1` then `d["a"] = 2`, the 1 is gone.
- As of Python 3.7+, dicts **preserve insertion order** (important for LRU Cache!).
- `d.get(key, default)` — safe lookup that won't throw KeyError.
- `d.setdefault(key, default)` — inserts default if key missing, then returns the value.
- All basic operations (`get`, `set`, `delete`, `in`) are **O(1) average**.


In [1]:
# ── Basic dict operations ──────────────────────────────────────────────────

d = {}

# Setting values — each key can only hold ONE value
d["hvac"] = "tech_001"
d["hvac"] = "tech_002"   # this OVERWRITES, not appends
print("Overwrite:", d)   # {'hvac': 'tech_002'} — tech_001 is gone!

# Safe lookup
print(d.get("plumbing", "not found"))   # 'not found' instead of KeyError
print(d.get("hvac", "not found"))       # 'tech_002'

# setdefault: "if key not there, set it; return whatever is there"
d.setdefault("plumbing", [])   # creates empty list at 'plumbing'
d.setdefault("hvac", [])       # hvac already exists, does NOT overwrite
print("After setdefault:", d)

# Checking membership
print("hvac" in d)      # True  — O(1)
print("roofing" in d)   # False — O(1)

# Iterating
d2 = {"a": 1, "b": 2, "c": 3}
for key, value in d2.items():
    print(f"  {key} -> {value}")


Overwrite: {'hvac': 'tech_002'}
not found
tech_002
After setdefault: {'hvac': 'tech_002', 'plumbing': []}
True
False
  a -> 1
  b -> 2
  c -> 3


### Why `dict` alone isn't enough for interviews

The problem: real-world data rarely fits the "one key → one value" model.

```
tech_001 can do → HVAC, Plumbing, Electrical
customer_A has  → property_1, property_2, property_3
job_99 needs    → part_A, part_B, part_A (duplicate parts)
```

When you need **one key → many values**, you have two choices:
1. Use `dict` with `list` as values (build it yourself → MultiMap)
2. Use `defaultdict(list)` as a shortcut

Let's look at the shortcut first.


---
## Part 2: The `collections` Module — Your Interview Toolkit

Python's `collections` module has four structures you should know cold.
They're all just dict subclasses or wrappers, but they save a lot of boilerplate.


In [2]:
from collections import defaultdict, Counter, OrderedDict, deque

# ── defaultdict ────────────────────────────────────────────────────────────
# Like a regular dict, but if you access a missing key, it auto-creates
# a value using the factory function you pass in.

skills = defaultdict(list)   # missing keys get an empty list

skills["tech_001"].append("HVAC")       # no KeyError even though key is new
skills["tech_001"].append("Plumbing")
skills["tech_002"].append("Electrical")

print("defaultdict:", dict(skills))
# {'tech_001': ['HVAC', 'Plumbing'], 'tech_002': ['Electrical']}

# Compare: doing this with a plain dict is clunky
plain = {}
for tech, skill in [("tech_001", "HVAC"), ("tech_001", "Plumbing")]:
    if tech not in plain:        # extra check required every time
        plain[tech] = []
    plain[tech].append(skill)
# defaultdict eliminates this pattern entirely


defaultdict: {'tech_001': ['HVAC', 'Plumbing'], 'tech_002': ['Electrical']}


In [3]:
# ── Counter ───────────────────────────────────────────────────────────────
# A dict subclass for counting. Keys are items, values are counts.
# Missing keys return 0 instead of KeyError.

job_types = ["HVAC", "Plumbing", "HVAC", "Electrical", "HVAC", "Plumbing"]
counts = Counter(job_types)
print("Counter:", counts)
# Counter({'HVAC': 3, 'Plumbing': 2, 'Electrical': 1})

# Most common: O(n log k) where k = number you want
print("Top 2:", counts.most_common(2))   # [('HVAC', 3), ('Plumbing', 2)]

# Useful in interviews: frequency analysis, finding modes


Counter: Counter({'HVAC': 3, 'Plumbing': 2, 'Electrical': 1})
Top 2: [('HVAC', 3), ('Plumbing', 2)]


In [4]:
# ── OrderedDict ───────────────────────────────────────────────────────────
# A dict that remembers insertion order AND lets you move items to front/back.
# Regular dicts preserve insertion order since Python 3.7, but OrderedDict
# has TWO extra methods that are critical for LRU Cache:
#   - move_to_end(key)           # move to back (most recently used)
#   - move_to_end(key, last=False) # move to front (least recently used)

od = OrderedDict()
od["first"] = 1
od["second"] = 2
od["third"] = 3
print("Original:", list(od.keys()))   # ['first', 'second', 'third']

od.move_to_end("first")               # move to back (mark as recently used)
print("After move 'first' to end:", list(od.keys()))   # ['second', 'third', 'first']

od.move_to_end("first", last=False)   # move to front (mark as least recently used)
print("After move 'first' to front:", list(od.keys()))  # ['first', 'second', 'third']

# popitem() removes from the END by default
key, val = od.popitem()
print("Popped from end:", key, val)   # 'third', 3

# popitem(last=False) removes from the FRONT = "evict least recently used"
key, val = od.popitem(last=False)
print("Popped from front:", key, val)  # 'first', 1


Original: ['first', 'second', 'third']
After move 'first' to end: ['second', 'third', 'first']
After move 'first' to front: ['first', 'second', 'third']
Popped from end: third 3
Popped from front: first 1


In [5]:
# ── deque (double-ended queue) ─────────────────────────────────────────────
# Not a dict, but pairs with dicts constantly in interview problems.
# Think of it as a list where append/pop from BOTH ends is O(1).
# Regular list: pop(0) is O(n). deque: popleft() is O(1).

dq = deque([1, 2, 3])
dq.appendleft(0)    # O(1) — add to front
dq.append(4)        # O(1) — add to back
print("deque:", dq)  # deque([0, 1, 2, 3, 4])

dq.popleft()        # O(1) — remove from front (list.pop(0) would be O(n)!)
print("After popleft:", dq)  # deque([1, 2, 3, 4])

# maxlen: auto-evicts oldest item when full — built-in sliding window!
window = deque(maxlen=3)
for i in range(6):
    window.append(i)
    print(f"  after append({i}): {list(window)}")


deque: deque([0, 1, 2, 3, 4])
After popleft: deque([1, 2, 3, 4])
  after append(0): [0]
  after append(1): [0, 1]
  after append(2): [0, 1, 2]
  after append(3): [1, 2, 3]
  after append(4): [2, 3, 4]
  after append(5): [3, 4, 5]


---
## Part 3: MultiMap — One Key, Many Values

**The problem `dict` doesn't solve:**
```
tech_001 → [HVAC, Plumbing, Electrical]   # one key, multiple values
```

A MultiMap is exactly this: a mapping where each key holds a **list** (or set) of values.

### The interface ServiceTitan asked for:
```
put(key, value)       — add a value to key's list
get(key)              — return all values for key
remove(key, value)    — remove one occurrence of value from key's list
keys()                — return all keys with at least one value
size()                — total number of (key, value) pairs
```

We'll build three versions, each improving on the last.


In [6]:
# ── Version 1: Naive (for understanding) ──────────────────────────────────
# Uses a plain dict with lists. Works, but has a subtle bug in remove().

class MultiMapV1:
    def __init__(self):
        self._data = {}   # key → [value1, value2, ...]

    def put(self, key, value):
        # If key doesn't exist yet, create an empty list first
        if key not in self._data:
            self._data[key] = []
        self._data[key].append(value)

    def get(self, key):
        # Return copy so caller can't mutate our internal list
        return list(self._data.get(key, []))

    def remove(self, key, value):
        if key not in self._data:
            return False
        try:
            self._data[key].remove(value)   # removes FIRST occurrence
            # Bug: if list becomes empty, the key still exists with [] value
            # keys() would return it even though it has no values
            return True
        except ValueError:
            return False

    def keys(self):
        # Bug: returns keys with empty lists too
        return list(self._data.keys())

    def size(self):
        return sum(len(v) for v in self._data.values())

# Test it
mm = MultiMapV1()
mm.put("colors", "red")
mm.put("colors", "blue")
mm.put("colors", "red")   # duplicates allowed
print("get:", mm.get("colors"))    # ['red', 'blue', 'red']
mm.remove("colors", "red")        # removes first 'red'
print("after remove:", mm.get("colors"))   # ['blue', 'red']

# Expose the bug
mm.remove("colors", "blue")
mm.remove("colors", "red")
print("keys after emptying:", mm.keys())   # ['colors'] — BUG: empty list still there
print("size:", mm.size())                  # 0 — size is correct though


get: ['red', 'blue', 'red']
after remove: ['blue', 'red']
keys after emptying: ['colors']
size: 0


In [7]:
# ── Version 2: Clean (interview-ready) ────────────────────────────────────
# Uses defaultdict to eliminate the "if key not in dict" pattern.
# Cleans up empty lists after remove().

from collections import defaultdict

class MultiMap:
    """
    A dict that maps each key to a list of values.
    
    Time complexity:
        put()    — O(1) amortized
        get()    — O(k) where k = number of values for that key (for the copy)
        remove() — O(k) to scan for value
        keys()   — O(n) where n = number of keys
        size()   — O(n) — we could make this O(1) with a counter if needed
    
    Space: O(n) total pairs stored.
    """
    
    def __init__(self):
        # defaultdict(list): accessing a missing key auto-creates an empty list
        # This eliminates the "if key not in self._data: self._data[key] = []" pattern
        self._data = defaultdict(list)
        self._size = 0   # track total pairs for O(1) size()
    
    def put(self, key, value):
        """Add a value to this key's list. Duplicates are allowed."""
        self._data[key].append(value)
        self._size += 1
    
    def get(self, key):
        """Return all values for key. Returns empty list if key doesn't exist."""
        # Return a copy — never expose internal mutable state
        return list(self._data[key])   # defaultdict returns [] for missing keys
    
    def remove(self, key, value):
        """
        Remove one occurrence of value from key's list.
        If the list becomes empty, also remove the key entirely.
        Returns True if value was found and removed, False otherwise.
        """
        if key not in self._data:
            return False
        
        lst = self._data[key]
        try:
            lst.remove(value)   # list.remove() removes FIRST occurrence, O(k)
            self._size -= 1
            
            # Clean up: if no values left, remove the key entirely
            # This fixes the V1 bug where keys() returned empty-list keys
            if not lst:
                del self._data[key]
            
            return True
        except ValueError:
            return False   # value wasn't in the list
    
    def keys(self):
        """Return all keys that have at least one value."""
        return list(self._data.keys())
    
    def size(self):
        """Total number of (key, value) pairs across all keys."""
        return self._size   # O(1) because we track it
    
    def __repr__(self):
        return f"MultiMap({dict(self._data)})"


# ── Full test suite ────────────────────────────────────────────────────────
print("=== MultiMap Tests ===")
mm = MultiMap()

# Basic put/get
mm.put("colors", "red")
mm.put("colors", "blue")
mm.put("colors", "red")    # duplicate
mm.put("shapes", "circle")

print("get colors:", mm.get("colors"))    # ['red', 'blue', 'red']
print("get shapes:", mm.get("shapes"))    # ['circle']
print("get missing:", mm.get("xyz"))      # []  — no KeyError
print("keys:", sorted(mm.keys()))         # ['colors', 'shapes']
print("size:", mm.size())                 # 4

# Remove
print("\n--- remove tests ---")
print("remove red:", mm.remove("colors", "red"))   # True
print("colors after:", mm.get("colors"))            # ['blue', 'red'] — first red removed
print("remove missing val:", mm.remove("colors", "green"))  # False
print("remove missing key:", mm.remove("xyz", "abc"))       # False

# Auto-cleanup when list empties
mm.remove("shapes", "circle")
print("keys after shapes emptied:", mm.keys())   # ['colors'] — 'shapes' gone

print("\nsize:", mm.size())   # 2


=== MultiMap Tests ===
get colors: ['red', 'blue', 'red']
get shapes: ['circle']
get missing: []
keys: ['colors', 'shapes', 'xyz']
size: 4

--- remove tests ---
remove red: True
colors after: ['blue', 'red']
remove missing val: False
remove missing key: False
keys after shapes emptied: ['colors', 'xyz']

size: 2


### Part 3b: Inverted MultiMap (likely Part 2 variant)

A very common Part 2 twist: "now add a method to look up by value instead of key."

This is called an **inverted index** — you maintain a second MultiMap that's the reverse
of the first. Trade space for lookup speed.

Real ST use case: "Given a skill, find all techs who have it." (forward = tech→skills, 
inverted = skill→techs)


In [8]:
# ── InvertedMultiMap ──────────────────────────────────────────────────────
# Supports BOTH directions:
#   get_by_key(key)     → all values for that key    (normal direction)
#   get_by_value(value) → all keys that have that value (inverted direction)
#
# We maintain TWO MultiMaps internally and keep them in sync.

class InvertedMultiMap:
    """
    Bidirectional MultiMap: look up by key OR by value.
    
    Example:
        imap.put("tech_001", "HVAC")
        imap.put("tech_002", "HVAC")
        imap.put("tech_001", "Plumbing")
        
        imap.get_by_key("tech_001")    → ["HVAC", "Plumbing"]
        imap.get_by_value("HVAC")      → ["tech_001", "tech_002"]
    
    Space: O(n) extra for the inverted index.
    All operations still O(k) where k = list length.
    """
    
    def __init__(self):
        self._forward  = defaultdict(list)   # key   → [values]
        self._inverted = defaultdict(list)   # value → [keys]
    
    def put(self, key, value):
        """Add key→value AND value→key to both indexes."""
        self._forward[key].append(value)
        self._inverted[value].append(key)
    
    def get_by_key(self, key):
        """All values for this key."""
        return list(self._forward[key])
    
    def get_by_value(self, value):
        """All keys that have this value."""
        return list(self._inverted[value])
    
    def remove(self, key, value):
        """Remove one key→value pair. Updates both indexes."""
        # Remove from forward index
        try:
            self._forward[key].remove(value)
            if not self._forward[key]:
                del self._forward[key]
        except ValueError:
            return False
        
        # Remove from inverted index (must stay in sync!)
        try:
            self._inverted[value].remove(key)
            if not self._inverted[value]:
                del self._inverted[value]
        except ValueError:
            pass   # shouldn't happen if data was consistent, but be safe
        
        return True


# ── Test it ────────────────────────────────────────────────────────────────
imap = InvertedMultiMap()
imap.put("tech_001", "HVAC")
imap.put("tech_001", "Plumbing")
imap.put("tech_002", "HVAC")
imap.put("tech_002", "Electrical")
imap.put("tech_003", "Plumbing")

print("tech_001's skills:", imap.get_by_key("tech_001"))      # ['HVAC', 'Plumbing']
print("who can do HVAC:", imap.get_by_value("HVAC"))          # ['tech_001', 'tech_002']
print("who can do Plumbing:", imap.get_by_value("Plumbing"))  # ['tech_001', 'tech_003']
print("who can do Roofing:", imap.get_by_value("Roofing"))    # [] — no KeyError

# Remove and verify BOTH directions updated
imap.remove("tech_001", "HVAC")
print("\nAfter removing tech_001's HVAC:")
print("tech_001's skills:", imap.get_by_key("tech_001"))   # ['Plumbing']
print("who can do HVAC:", imap.get_by_value("HVAC"))       # ['tech_002'] — tech_001 gone


tech_001's skills: ['HVAC', 'Plumbing']
who can do HVAC: ['tech_001', 'tech_002']
who can do Plumbing: ['tech_001', 'tech_003']
who can do Roofing: []

After removing tech_001's HVAC:
tech_001's skills: ['Plumbing']
who can do HVAC: ['tech_002']


---
## Part 4: LRU Cache (Least Recently Used)

**The problem:** You have expensive data (DB queries, API calls). You want to cache results,
but you can't cache everything forever. So you evict the item that was **used longest ago**.

**Real ST use case:** Caching tenant configuration lookups. 12,000 tenants, most traffic
is concentrated in a few hundred. Keep hot tenants in fast cache; evict cold ones.

### The invariant we must maintain at all times:
> The item at the "front" of our ordering is the Least Recently Used (LRU).
> The item at the "back" is the Most Recently Used (MRU).
> On get/put, move that item to the back.
> On eviction, remove from the front.

### Why `dict` alone doesn't work:
- We need O(1) lookup by key → dict ✓
- We need O(1) move-to-end (after every access) → regular dict can't do this
- We need O(1) delete-from-front (on eviction) → regular list is O(n)

### Solution: OrderedDict
Python's `OrderedDict` maintains insertion order AND has `move_to_end()`.
This makes it perfect for LRU with no extra data structures.


In [9]:
from collections import OrderedDict

# ── LRU Cache with OrderedDict ─────────────────────────────────────────────
class LRUCache:
    """
    Fixed-capacity cache. On overflow, evicts the least recently used item.
    
    Both get() and put() count as a 'use' — they both move item to MRU position.
    
    Time:  O(1) for get and put  (OrderedDict operations are O(1))
    Space: O(capacity)
    """
    
    def __init__(self, capacity: int):
        if capacity <= 0:
            raise ValueError("Capacity must be positive")
        self.capacity = capacity
        # OrderedDict preserves insertion order
        # We'll use: FRONT = LRU (oldest), BACK = MRU (newest)
        self._cache = OrderedDict()
    
    def get(self, key):
        """
        Return value for key, or -1 if not present.
        SIDE EFFECT: marks this key as most recently used.
        """
        if key not in self._cache:
            return -1
        
        # Move to end = mark as most recently used
        self._cache.move_to_end(key)
        return self._cache[key]
    
    def put(self, key, value):
        """
        Insert or update key→value.
        If key exists: update value, mark as MRU.
        If new key and at capacity: evict LRU first, then insert.
        """
        if key in self._cache:
            # Update existing — also marks as MRU
            self._cache.move_to_end(key)
            self._cache[key] = value
        else:
            # New key
            if len(self._cache) >= self.capacity:
                # popitem(last=False) removes from FRONT = evicts LRU
                evicted_key, evicted_val = self._cache.popitem(last=False)
                print(f"  [evicted] {evicted_key}={evicted_val}")
            
            # Insert at end = MRU position
            self._cache[key] = value
    
    def __repr__(self):
        # Print in LRU → MRU order (front to back)
        items = list(self._cache.items())
        order = " → ".join(f"{k}:{v}" for k, v in items)
        return f"LRUCache[{order}] (LRU←  →MRU)"


# ── Walkthrough — trace every step ────────────────────────────────────────
print("=== LRU Cache (capacity=3) ===\n")
lru = LRUCache(capacity=3)

lru.put("tenant_A", "config_A")
print("put A:", lru)

lru.put("tenant_B", "config_B")
print("put B:", lru)

lru.put("tenant_C", "config_C")
print("put C:", lru)
print("Cache is now FULL")

print("\nget A:", lru.get("tenant_A"))   # Hits! Moves A to MRU
print("order:", lru)                       # B is now LRU (oldest unused)

print("\nput D: (should evict B, the LRU)")
lru.put("tenant_D", "config_D")
print("order:", lru)

print("\nget B:", lru.get("tenant_B"))   # -1, was evicted


=== LRU Cache (capacity=3) ===

put A: LRUCache[tenant_A:config_A] (LRU←  →MRU)
put B: LRUCache[tenant_A:config_A → tenant_B:config_B] (LRU←  →MRU)
put C: LRUCache[tenant_A:config_A → tenant_B:config_B → tenant_C:config_C] (LRU←  →MRU)
Cache is now FULL

get A: config_A
order: LRUCache[tenant_B:config_B → tenant_C:config_C → tenant_A:config_A] (LRU←  →MRU)

put D: (should evict B, the LRU)
  [evicted] tenant_B=config_B
order: LRUCache[tenant_C:config_C → tenant_A:config_A → tenant_D:config_D] (LRU←  →MRU)

get B: -1


In [10]:
# ── LRU Cache: Linked List version (what they ask you to implement) ────────
# The OrderedDict version above is great for production Python code.
# But interviewers sometimes say "don't use OrderedDict" and want you to
# build the underlying mechanism yourself.
#
# The mechanism: dict + doubly linked list
#   dict[key] → pointer to node in linked list
#   linked list maintains order: head=LRU, tail=MRU
#   moving a node to tail is O(1) if you have a pointer to it

class DLinkedNode:
    """A node in a doubly linked list."""
    def __init__(self, key=None, val=None):
        self.key  = key
        self.val  = val
        self.prev = None   # pointer to previous node
        self.next = None   # pointer to next node

class LRUCacheManual:
    """
    LRU Cache using dict + doubly linked list — no OrderedDict.
    
    Structure:
        dummy_head ↔ [LRU] ↔ ... ↔ [MRU] ↔ dummy_tail
        
    Dummy head/tail nodes simplify edge cases (no special-casing for empty list).
    dict maps key → node for O(1) access.
    """
    
    def __init__(self, capacity):
        self.capacity = capacity
        self._cache = {}   # key → DLinkedNode
        
        # Sentinel nodes — never hold real data, just simplify insert/remove logic
        self._head = DLinkedNode()   # dummy head — next is LRU
        self._tail = DLinkedNode()   # dummy tail — prev is MRU
        self._head.next = self._tail
        self._tail.prev = self._head
    
    def _remove(self, node):
        """Detach a node from the linked list. O(1)."""
        node.prev.next = node.next
        node.next.prev = node.prev
    
    def _add_to_tail(self, node):
        """Insert node just before dummy tail (= MRU position). O(1)."""
        node.prev = self._tail.prev
        node.next = self._tail
        self._tail.prev.next = node
        self._tail.prev = node
    
    def get(self, key):
        if key not in self._cache:
            return -1
        node = self._cache[key]
        # Move to MRU: remove from current position, add to tail
        self._remove(node)
        self._add_to_tail(node)
        return node.val
    
    def put(self, key, value):
        if key in self._cache:
            node = self._cache[key]
            node.val = value
            self._remove(node)
            self._add_to_tail(node)
        else:
            if len(self._cache) >= self.capacity:
                # Evict LRU = node right after dummy head
                lru_node = self._head.next
                self._remove(lru_node)
                del self._cache[lru_node.key]
            
            new_node = DLinkedNode(key, value)
            self._add_to_tail(new_node)
            self._cache[key] = new_node
    
    def order(self):
        """Return keys in LRU→MRU order (for debugging)."""
        keys = []
        node = self._head.next
        while node != self._tail:
            keys.append(node.key)
            node = node.next
        return keys


# Test — same behavior as OrderedDict version
print("=== Manual LRU (dict + linked list) ===\n")
lru2 = LRUCacheManual(capacity=3)
lru2.put("A", 1); lru2.put("B", 2); lru2.put("C", 3)
print("order:", lru2.order())    # ['A', 'B', 'C']

lru2.get("A")                    # access A → moves to MRU
print("after get A:", lru2.order())   # ['B', 'C', 'A']

lru2.put("D", 4)                 # evicts B (LRU)
print("after put D:", lru2.order())   # ['C', 'A', 'D']
print("get B:", lru2.get("B"))        # -1 — evicted


=== Manual LRU (dict + linked list) ===

order: ['A', 'B', 'C']
after get A: ['B', 'C', 'A']
after put D: ['C', 'A', 'D']
get B: -1


---
## Part 5: LFU Cache (Least Frequently Used)

**How it differs from LRU:**
- LRU evicts the item **unused for the longest time** (recency)
- LFU evicts the item **accessed fewest times** (frequency)
- Tiebreaker: when two items have equal frequency, evict the **least recently used** of them

**When LFU is better:** If some items are rare but recent (they'd survive an LRU cache),
and other items are accessed constantly, LFU keeps the high-frequency items even if they
haven't been touched recently.

### The data structure trick:
We maintain:
1. `key_to_val`: key → value
2. `key_to_freq`: key → frequency count
3. `freq_to_keys`: frequency → OrderedDict of keys (preserves insertion order for LRU tiebreak)
4. `min_freq`: track current minimum frequency for O(1) eviction


In [11]:
from collections import defaultdict, OrderedDict

class LFUCache:
    """
    Fixed-capacity cache with Least Frequently Used eviction.
    Tiebreak: among equal-frequency items, evict the least recently used.
    
    Time:  O(1) for get and put
    Space: O(capacity)
    """
    
    def __init__(self, capacity: int):
        self.capacity  = capacity
        self.min_freq  = 0
        
        self._key_to_val  = {}                    # key → value
        self._key_to_freq = {}                    # key → access count
        self._freq_to_keys = defaultdict(OrderedDict)  
        # freq → OrderedDict{key: None}
        # OrderedDict gives us LRU tiebreaking within each frequency bucket
        # (oldest key is first, newest is last)
    
    def _increment_freq(self, key):
        """Move key from its current frequency bucket to the next one."""
        freq = self._key_to_freq[key]
        
        # Remove from current bucket
        del self._freq_to_keys[freq][key]
        
        # If current bucket empty AND it was the minimum, bump min_freq up
        if not self._freq_to_keys[freq] and freq == self.min_freq:
            self.min_freq += 1
        
        # Add to next bucket
        new_freq = freq + 1
        self._key_to_freq[key] = new_freq
        self._freq_to_keys[new_freq][key] = None
    
    def get(self, key):
        if key not in self._key_to_val:
            return -1
        self._increment_freq(key)   # accessing counts as use
        return self._key_to_val[key]
    
    def put(self, key, value):
        if self.capacity == 0:
            return
        
        if key in self._key_to_val:
            # Update value and frequency
            self._key_to_val[key] = value
            self._increment_freq(key)
        else:
            if len(self._key_to_val) >= self.capacity:
                # Evict: go to min_freq bucket, remove oldest (first) key
                bucket = self._freq_to_keys[self.min_freq]
                evicted_key, _ = bucket.popitem(last=False)   # FIFO within bucket
                del self._key_to_val[evicted_key]
                del self._key_to_freq[evicted_key]
                print(f"  [evicted] {evicted_key}")
            
            # Insert new key at frequency 1
            self._key_to_val[key] = value
            self._key_to_freq[key] = 1
            self._freq_to_keys[1][key] = None
            self.min_freq = 1   # new items always start at freq 1
    
    def state(self):
        """Debug: show frequency buckets."""
        buckets = {}
        for freq, od in self._freq_to_keys.items():
            if od:
                buckets[freq] = list(od.keys())
        return f"min_freq={self.min_freq}, buckets={buckets}"


# ── Walkthrough ────────────────────────────────────────────────────────────
print("=== LFU Cache (capacity=3) ===\n")
lfu = LFUCache(3)

lfu.put("A", 1); lfu.put("B", 2); lfu.put("C", 3)
print("state:", lfu.state())     # all freq=1

lfu.get("A")   # A now freq=2
lfu.get("A")   # A now freq=3
lfu.get("B")   # B now freq=2
print("state:", lfu.state())     # A:3, B:2, C:1

print("\nput D — should evict C (only freq=1 item)")
lfu.put("D", 4)
print("state:", lfu.state())

print("\nget C:", lfu.get("C"))   # -1 — was evicted

print("\nput E — B and D both have freq=1 (or 2), check tiebreak")
lfu.put("E", 5)
print("state:", lfu.state())


=== LFU Cache (capacity=3) ===

state: min_freq=1, buckets={1: ['A', 'B', 'C']}
state: min_freq=1, buckets={1: ['C'], 2: ['B'], 3: ['A']}

put D — should evict C (only freq=1 item)
  [evicted] C
state: min_freq=1, buckets={1: ['D'], 2: ['B'], 3: ['A']}

get C: -1

put E — B and D both have freq=1 (or 2), check tiebreak
  [evicted] D
state: min_freq=1, buckets={1: ['E'], 2: ['B'], 3: ['A']}


---
## Part 6: Priority-Queue-Backed Dict (for Dispatch / Scheduling)

**The problem:** You have a queue of jobs with priorities. You want:
- Fast enqueue (add a job)
- Fast dequeue of HIGHEST priority job
- Fast lookup: "is job X still in queue? What's its priority?"
- Fast update: "job X's priority just changed"

This comes up in ServiceTitan dispatch: "assign the highest priority unassigned job to the
next available tech."

### The structure: `dict` + `heapq`

Python's `heapq` is a **min-heap** — `heappop()` gives you the smallest item.
For a max-priority queue, negate priorities.

The challenge: heaps don't support update or removal by key. The solution is "lazy deletion":
mark stale entries and skip them on pop.


In [12]:
import heapq

class PriorityDict:
    """
    A dict where each key has a priority.
    Supports O(log n) push, O(log n) pop_min, O(1) lookup, O(log n) update.
    
    Internally: dict (key→priority) + heap for ordering.
    Lazy deletion: when priority changes, old heap entry becomes 'stale'.
    Stale entries are skipped on pop.
    """
    
    def __init__(self):
        self._heap   = []          # min-heap of (priority, key)
        self._dict   = {}          # key → current priority (source of truth)
        # Note: _heap may contain stale (old_priority, key) entries
        #       An entry is stale if _dict[key] != old_priority
    
    def push(self, key, priority):
        """Add or update key with given priority. O(log n)."""
        self._dict[key] = priority
        heapq.heappush(self._heap, (priority, key))
        # If key existed before, old heap entry is now stale — will be skipped on pop
    
    def pop_min(self):
        """Remove and return (key, priority) with lowest priority value. O(log n)."""
        while self._heap:
            priority, key = heapq.heappop(self._heap)
            
            # Skip stale entries: if _dict says key has a different priority,
            # this heap entry is old — ignore it
            if key in self._dict and self._dict[key] == priority:
                del self._dict[key]
                return key, priority
        
        raise KeyError("pop from empty PriorityDict")
    
    def peek_min(self):
        """Return (key, priority) with lowest priority without removing. O(log n) amortized."""
        while self._heap:
            priority, key = self._heap[0]
            if key in self._dict and self._dict[key] == priority:
                return key, priority
            heapq.heappop(self._heap)   # discard stale
        raise KeyError("empty PriorityDict")
    
    def remove(self, key):
        """Remove a key entirely. O(1) — just delete from dict; heap entry becomes stale."""
        if key not in self._dict:
            raise KeyError(key)
        del self._dict[key]
        # Don't touch heap — stale entry will be skipped lazily on next pop
    
    def __contains__(self, key):
        return key in self._dict
    
    def __len__(self):
        return len(self._dict)   # heap may be larger due to stale entries
    
    def priority_of(self, key):
        """Look up current priority of a key. O(1)."""
        return self._dict[key]


# ── Dispatch simulation ────────────────────────────────────────────────────
print("=== Priority-Based Job Dispatch ===\n")
# Lower number = higher priority (emergency < urgent < routine)
EMERGENCY = 1
URGENT = 2
ROUTINE = 3

dispatch_queue = PriorityDict()
dispatch_queue.push("job_001", ROUTINE)
dispatch_queue.push("job_002", URGENT)
dispatch_queue.push("job_003", EMERGENCY)
dispatch_queue.push("job_004", ROUTINE)

print(f"Queue has {len(dispatch_queue)} jobs")
print(f"Next job: {dispatch_queue.peek_min()}")

# Dispatch jobs in priority order
print("\nDispatching:")
while len(dispatch_queue) > 0:
    job, priority = dispatch_queue.pop_min()
    label = {1: "EMERGENCY", 2: "URGENT", 3: "ROUTINE"}[priority]
    print(f"  → {job} ({label})")

print("\n--- Priority update test ---")
q = PriorityDict()
q.push("job_A", ROUTINE)
q.push("job_B", ROUTINE)
print("Before escalation:", q.peek_min())

# Customer escalated job_B to emergency
q.push("job_B", EMERGENCY)   # update — old ROUTINE entry becomes stale
print("After escalating job_B:", q.peek_min())   # job_B should be first now

job, pri = q.pop_min()
print(f"Popped: {job} (priority={pri})")
job, pri = q.pop_min()
print(f"Popped: {job} (priority={pri})")   # job_A's stale entry was skipped


=== Priority-Based Job Dispatch ===

Queue has 4 jobs
Next job: ('job_003', 1)

Dispatching:
  → job_003 (EMERGENCY)
  → job_002 (URGENT)
  → job_001 (ROUTINE)
  → job_004 (ROUTINE)

--- Priority update test ---
Before escalation: ('job_A', 3)
After escalating job_B: ('job_B', 1)
Popped: job_B (priority=1)
Popped: job_A (priority=3)


---
## Part 7: Thread-Safe Wrappers (Part 2 Pattern)

**Why this comes up:** ServiceTitan's HackerRank Part 2 often adds a concurrency requirement
on top of the Part 1 data structure. They want to know if you understand race conditions.

**The simplest correct solution:** Wrap all mutations in a `threading.Lock()`.

**What a race condition looks like without a lock:**
```
Thread A reads size=5
Thread B reads size=5
Thread A writes size=6
Thread B writes size=6   ← WRONG: should be 7
```

**For interviews:** Adding a lock and briefly explaining the tradeoff
(coarse-grained lock = easy to reason about, lower throughput vs reader-writer lock)
is usually enough to satisfy the requirement.


In [13]:
import threading

class ThreadSafeMultiMap:
    """
    Thread-safe MultiMap. Uses a single lock for all operations.
    
    Tradeoffs:
        + Simple: easy to verify correctness
        + One lock: no deadlock risk (single resource)
        - Coarse-grained: all operations block each other, even concurrent reads
    
    For higher throughput: use threading.RLock for re-entrant locking,
    or a readers-writer lock pattern (many concurrent reads, exclusive writes).
    """
    
    def __init__(self):
        self._data = defaultdict(list)
        self._size = 0
        self._lock = threading.Lock()
        # threading.Lock() is a mutual exclusion lock:
        # only one thread can hold it at a time
        # Other threads block on acquire() until it's released
    
    def put(self, key, value):
        with self._lock:    # "with" auto-releases even if exception occurs
            self._data[key].append(value)
            self._size += 1
    
    def get(self, key):
        with self._lock:
            return list(self._data[key])
    
    def remove(self, key, value):
        with self._lock:
            if key not in self._data:
                return False
            try:
                self._data[key].remove(value)
                self._size -= 1
                if not self._data[key]:
                    del self._data[key]
                return True
            except ValueError:
                return False
    
    def size(self):
        with self._lock:
            return self._size


# ── Test with concurrent threads ───────────────────────────────────────────
import time

ts_mm = ThreadSafeMultiMap()
errors = []

def writer(thread_id, count=50):
    for i in range(count):
        ts_mm.put(f"key_{i % 5}", f"val_t{thread_id}_{i}")

def reader(count=50):
    for i in range(count):
        result = ts_mm.get(f"key_{i % 5}")
        # Just checking it doesn't throw

threads = [threading.Thread(target=writer, args=(i,)) for i in range(4)]
threads += [threading.Thread(target=reader) for _ in range(2)]

for t in threads: t.start()
for t in threads: t.join()

print(f"Final size: {ts_mm.size()}")
print(f"Keys: {sorted(ts_mm._data.keys())}")
print("No exceptions — thread safety confirmed")


Final size: 200
Keys: ['key_0', 'key_1', 'key_2', 'key_3', 'key_4']
No exceptions — thread safety confirmed


---
## Part 8: When to Use What — Interview Cheatsheet

| Structure | Use When | Key Operations | Complexity |
|---|---|---|---|
| `dict` | One key → one value | get, set, del | O(1) |
| `defaultdict(list)` | One key → many values, no boilerplate | same as dict | O(1) |
| `Counter` | Counting occurrences | most_common(k) | O(n log k) |
| `OrderedDict` | Need insertion-order + move_to_end | move_to_end, popitem | O(1) |
| **MultiMap** | One key → ordered list of values | put, get, remove | O(k) |
| **InvertedMultiMap** | Look up by key OR by value | get_by_key, get_by_value | O(k) |
| **LRU Cache** | Cache with recency eviction | get, put | O(1) |
| **LFU Cache** | Cache with frequency eviction | get, put | O(1) |
| **PriorityDict** | Queue where priority can change | push, pop_min, remove | O(log n) |

### How to answer "what's the time complexity?" in an interview

Walk through each method:
1. **State the complexity**: "get() is O(k) where k is the number of values for that key"
2. **Explain why**: "because we copy the list to return it"  
3. **Note the tradeoff if relevant**: "we could return a view (O(1)) but then the caller could mutate our internal state"

### Common edge cases to handle (always test these)

- Empty structure: `get()` on empty MultiMap should return `[]`, not throw
- Missing key: `get("nonexistent")` should return default, not KeyError
- Duplicate values: `put("a", "x"); put("a", "x")` — both should be stored
- Last value removed: after `remove()` empties a key's list, that key should disappear from `keys()`
- Capacity 1 in LRU: every `put` evicts the previous item
- LFU tie: two items with same frequency → evict the older one


In [14]:
# ── Quick sanity check — all structures in 20 lines ───────────────────────
print("=== Final Sanity Check ===\n")

# MultiMap
mm = MultiMap()
mm.put("a", 1); mm.put("a", 2); mm.put("b", 3)
assert mm.get("a") == [1, 2]
assert mm.get("missing") == []
assert mm.size() == 3
print("MultiMap ✓")

# LRU
lru = LRUCache(2)
lru.put("x", 10); lru.put("y", 20)
assert lru.get("x") == 10
lru.put("z", 30)           # evicts y (LRU)
assert lru.get("y") == -1  # gone
assert lru.get("x") == 10  # still there (was accessed after y)
print("LRU Cache ✓")

# LFU
lfu = LFUCache(2)
lfu.put("a", 1); lfu.put("b", 2)
lfu.get("a"); lfu.get("a")  # a freq=3
lfu.put("c", 3)              # evicts b (lower freq)
assert lfu.get("b") == -1
assert lfu.get("a") == 1
print("LFU Cache ✓")

# PriorityDict
pd = PriorityDict()
pd.push("job1", 3); pd.push("job2", 1); pd.push("job3", 2)
k, p = pd.pop_min()
assert k == "job2" and p == 1
print("PriorityDict ✓")

print("\nAll structures verified.")


=== Final Sanity Check ===

MultiMap ✓
  [evicted] y=20
LRU Cache ✓
  [evicted] b
LFU Cache ✓
PriorityDict ✓

All structures verified.
